In [1]:
import copy
import itertools
import os
import random
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model
from kryptonet import load_kryptonet

TARGET = 0.90


def load(name):
    """Load (X, y) arrays for a dataset."""
    return np.load(f"Datasets/{name}-X.npy"), np.load(f"Datasets/{name}-y.npy")

## Feature selection

In [2]:
def lookup_table_acc(S_train, y_train, S_val, y_val, cols):
    """Val accuracy of a majority-vote lookup table over the bits in `cols`."""
    w = 2 ** np.arange(len(cols))
    idx_tr = ((S_train[:, cols] > 0) * w).sum(1)
    idx_va = ((S_val[:, cols] > 0) * w).sum(1)
    count = np.bincount(idx_tr, minlength=2 ** len(cols))
    pos = np.bincount(idx_tr, weights=y_train, minlength=2 ** len(cols))
    table = (2 * pos > count).astype(int)  # unseen cells default to 0
    return (table[idx_va] == y_val).mean()


def select_features(S_train, y_train, S_val, y_val, min_gain=0.005):
    """Greedy forward selection that may add one bit or a pair of bits per step."""
    n = S_train.shape[1]
    score = lambda cols: lookup_table_acc(S_train, y_train, S_val, y_val, list(cols))
    selected, best = [], 0.5
    while True:
        rest = [j for j in range(n) if j not in selected]
        cands = [[j] for j in rest] + [list(p) for p in itertools.combinations(rest, 2)]
        acc, add = max((score(selected + c), c) for c in cands)
        if acc <= best + min_gain:
            break
        selected, best = selected + add, acc
        print(f"  + {add} -> {sorted(selected)}  val acc {acc:.4f}")
    return sorted(selected)


def binarise(X):
    """Map each noisy +-1 feature to its sign."""
    return np.where(X > 0, 1.0, -1.0)


def mask_features(X, cols):
    """Zero out every feature not in `cols`, so KryptoNet still receives 40 inputs."""
    M = np.zeros_like(X)
    M[:, cols] = X[:, cols]
    return M

## LoRA on Kryptonite-40 with selected, binarised features

In [3]:
SEED = 42
RANKS = [1, 2, 4, 8, 16, 32, 64, 128, 256]
WEIGHT_DIR = "./weights/lora_fs_bin"


def set_seed(seed=42):
    """Set a seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_tensors(X, y):
    """Convert numpy arrays to float tensors."""
    return torch.from_numpy(X).float(), torch.from_numpy(y).float()


def accuracy(net, X, y):
    """Get accuracy of `net` on `X` and `y`."""
    net.eval()
    with torch.no_grad():
        return float(((net(X).squeeze(1) > 0).float() == y).float().mean())


# Adapted from MockPaperExperiments.ipynb
def fit(net, Xa, ya, Xv, yv, params, lr, epochs=300, patience=40, bs=128):
    """Adam on `params` with early stopping on a 20% validation split; restores the best weights."""
    opt = torch.optim.Adam(params, lr=lr)
    best, state, since = -1.0, None, 0
    for _ in range(epochs):
        net.train(); p = torch.randperm(len(ya))
        for i in range(0, len(ya), bs):
            bi = p[i:i + bs]
            loss = torch.nn.functional.binary_cross_entropy_with_logits(net(Xa[bi]).squeeze(1), ya[bi])
            opt.zero_grad(); loss.backward(); opt.step()
        va = accuracy(net, Xv, yv)
        if va > best:
            best, since, state = va, 0, copy.deepcopy(net.state_dict())
        else:
            since += 1
            if since >= patience:
                break
    net.load_state_dict(state)
    return net


def make_lora(rank):
    """Wrap the pretrained KryptoNet-40 with LoRA adapters of the given rank."""
    config = LoraConfig(
        r=rank,
        lora_alpha=2 * rank,
        target_modules=[str(n) for n in range(0, 9, 2)],  # linear layers: 0, 2, 4, 6, 8
        lora_dropout=0.05,
        bias="none",
    )
    return get_peft_model(load_kryptonet("kryptonet40.pt"), config)  # freezes the base weights


def split(X, y):
    """60% train, 20% validation, 20% test."""
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)
    return X_train, y_train, X_val, y_val, X_test, y_test


def rank_sweep(frac, ranks=RANKS, lr=1e-3):
    """Pick the best rank of LoRA on val of the first `frac`."""
    na, nv = int(round(frac * len(y_train))), int(round(frac * len(y_val)))
    print(f"=== {frac:.0%} of train/val: {na} train / {nv} val / {len(y_test)} test ===")

    # Feature selection on this subset's train/val only, then binarise and mask all splits with it
    cols = select_features(X_train[:na], y_train[:na], X_val[:nv], y_val[:nv])
    print(f"selected {len(cols)} features: {cols}")
    Xa, ya = to_tensors(mask_features(binarise(X_train[:na]), cols), y_train[:na])
    Xv, yv = to_tensors(mask_features(binarise(X_val[:nv]), cols), y_val[:nv])
    Xt, yt = to_tensors(mask_features(binarise(X_test), cols), y_test)
    out_dir = os.path.join(WEIGHT_DIR, f"{round(frac * 100)}pct")
    os.makedirs(out_dir, exist_ok=True)

    rows, best = [], None
    for r in ranks:
        set_seed(SEED)
        model = make_lora(r)
        trainable = [p for p in model.parameters() if p.requires_grad]
        model = fit(model, Xa, ya, Xv, yv, trainable, lr=lr)
        row = {"frac": frac, "features": cols, "rank": r, "params": sum(p.numel() for p in trainable),
               "train": accuracy(model, Xa, ya), "val": accuracy(model, Xv, yv)}
        rows.append(row)
        print(f"rank {r:>3} | params {row['params']:>9,} | train {row['train']:.4f} | val {row['val']:.4f}")
        if best is None or row["val"] > best["val"]:
            best, best_model = row, copy.deepcopy(model).merge_and_unload()

    # Save the best model for this fraction
    row = best
    row["test"] = accuracy(best_model, Xt, yt)
    row["state"] = best_model.state_dict()
    torch.save(row["state"], os.path.join(out_dir, "best.pt"))
    np.save(os.path.join(out_dir, "features.npy"), np.array(cols))
    print(f"Best for {frac:.0%}: rank {row['rank']} (val {row['val']:.4f}) -> test {row['test']:.4f}, "
          f"saved {os.path.join(out_dir, 'best.pt')}")
    return rows


set_seed(SEED)
X, y = load("kryptonite-40")
X_train, y_train, X_val, y_val, X_test, y_test = split(X, y)
results = {}

## 100% of data

In [4]:
results[1.0] = rank_sweep(1.0)

=== 100% of train/val: 19200 train / 6400 val / 6400 test ===
  + [6, 25] -> [6, 25]  val acc 0.5687
  + [24, 30] -> [6, 24, 25, 30]  val acc 0.6072
  + [12, 15] -> [6, 12, 15, 24, 25, 30]  val acc 0.6836
  + [7, 36] -> [6, 7, 12, 15, 24, 25, 30, 36]  val acc 0.7873
  + [21, 39] -> [6, 7, 12, 15, 21, 24, 25, 30, 36, 39]  val acc 0.9569
selected 10 features: [6, 7, 12, 15, 21, 24, 25, 30, 36, 39]
rank   1 | params     4,137 | train 0.9548 | val 0.9511
rank   2 | params     8,274 | train 0.9603 | val 0.9569
rank   4 | params    16,548 | train 0.9603 | val 0.9569
rank   8 | params    33,096 | train 0.9603 | val 0.9569
rank  16 | params    66,192 | train 0.9603 | val 0.9569
rank  32 | params   132,384 | train 0.9603 | val 0.9569
rank  64 | params   264,768 | train 0.9603 | val 0.9569
rank 128 | params   529,536 | train 0.9603 | val 0.9569
rank 256 | params 1,059,072 | train 0.9603 | val 0.9569
Best for 100%: rank 2 (val 0.9569) -> test 0.9631, saved ./weights/lora_fs_bin\100pct\best.pt


## 50% of data

In [5]:
results[0.5] = rank_sweep(0.5)

=== 50% of train/val: 9600 train / 3200 val / 6400 test ===
  + [6, 25] -> [6, 25]  val acc 0.5737
  + [12, 15] -> [6, 12, 15, 25]  val acc 0.6138
  + [24, 30] -> [6, 12, 15, 24, 25, 30]  val acc 0.6841
  + [21, 39] -> [6, 12, 15, 21, 24, 25, 30, 39]  val acc 0.7837
  + [7, 36] -> [6, 7, 12, 15, 21, 24, 25, 30, 36, 39]  val acc 0.9556
selected 10 features: [6, 7, 12, 15, 21, 24, 25, 30, 36, 39]
rank   1 | params     4,137 | train 0.9575 | val 0.9519
rank   2 | params     8,274 | train 0.9590 | val 0.9563
rank   4 | params    16,548 | train 0.9590 | val 0.9563
rank   8 | params    33,096 | train 0.9590 | val 0.9563
rank  16 | params    66,192 | train 0.9586 | val 0.9563
rank  32 | params   132,384 | train 0.9590 | val 0.9563
rank  64 | params   264,768 | train 0.9590 | val 0.9563
rank 128 | params   529,536 | train 0.9590 | val 0.9563
rank 256 | params 1,059,072 | train 0.9590 | val 0.9563
Best for 50%: rank 2 (val 0.9563) -> test 0.9631, saved ./weights/lora_fs_bin\50pct\best.pt


## 25% of data

In [6]:
results[0.25] = rank_sweep(0.25)

=== 25% of train/val: 4800 train / 1600 val / 6400 test ===
  + [6, 25] -> [6, 25]  val acc 0.5587
  + [12, 15] -> [6, 12, 15, 25]  val acc 0.6019
  + [24, 30] -> [6, 12, 15, 24, 25, 30]  val acc 0.6656
  + [21, 39] -> [6, 12, 15, 21, 24, 25, 30, 39]  val acc 0.7775
  + [7, 36] -> [6, 7, 12, 15, 21, 24, 25, 30, 36, 39]  val acc 0.9363
selected 10 features: [6, 7, 12, 15, 21, 24, 25, 30, 36, 39]
rank   1 | params     4,137 | train 0.9429 | val 0.9212
rank   2 | params     8,274 | train 0.9563 | val 0.9488
rank   4 | params    16,548 | train 0.9544 | val 0.9413
rank   8 | params    33,096 | train 0.9558 | val 0.9413
rank  16 | params    66,192 | train 0.9531 | val 0.9394
rank  32 | params   132,384 | train 0.9569 | val 0.9506
rank  64 | params   264,768 | train 0.9571 | val 0.9500
rank 128 | params   529,536 | train 0.9571 | val 0.9500
rank 256 | params 1,059,072 | train 0.9569 | val 0.9525
Best for 25%: rank 256 (val 0.9525) -> test 0.9631, saved ./weights/lora_fs_bin\25pct\best.pt


### Summary

In [7]:
print(f"{'subset':>6} | {'#feat':>5} | {'best rank':>9} | {'train':>6} | {'val':>6} | {'test':>6}")
for frac, rows in results.items():
    b = next(r for r in rows if "state" in r)
    print(f"{frac:>6.0%} | {len(b['features']):>5} | {b['rank']:>9} | {b['train']:.4f} | {b['val']:.4f} | {b['test']:.4f}")

# The best across subsets chosen by validation accuracy among each subset's ranks
overall = max((r for rows in results.values() for r in rows if "state" in r), key=lambda r: r["val"])
torch.save(overall["state"], os.path.join(WEIGHT_DIR, "best.pt"))
np.save(os.path.join(WEIGHT_DIR, "features.npy"), np.array(overall["features"]))
print()
print(f"Overall best: {overall['frac']:.0%} subset, rank {overall['rank']}, features {overall['features']} "
      f"(val {overall['val']:.4f}, test {overall['test']:.4f}) -> saved {os.path.join(WEIGHT_DIR, 'best.pt')}")

subset | #feat | best rank |  train |    val |   test
  100% |    10 |         2 | 0.9603 | 0.9569 | 0.9631
   50% |    10 |         2 | 0.9590 | 0.9563 | 0.9631
   25% |    10 |       256 | 0.9569 | 0.9525 | 0.9631

Overall best: 100% subset, rank 2, features [6, 7, 12, 15, 21, 24, 25, 30, 36, 39] (val 0.9569, test 0.9631) -> saved ./weights/lora_fs_bin\best.pt


## LoRA on Kryptonite-40-Adapt

In [8]:
def rank_sweep_lite(dataset, keyword, ranks=RANKS, lr=1e-3):
    """Lite version of rank sweep"""
    Xd, yd = load(dataset)
    Xa, ya, Xv, yv, Xte, yte = split(Xd, yd)
    print(f"=== {dataset}: {len(ya)} train / {len(yv)} val / {len(yte)} test ===")

    # Feature selection on this dataset's train/val only, then binarise and mask all splits with it
    cols = select_features(Xa, ya, Xv, yv)
    print(f"selected {len(cols)} features: {cols}")
    Xa, Xv, Xte = (mask_features(binarise(A), cols) for A in (Xa, Xv, Xte))
    (Xa, ya), (Xv, yv), (Xte, yte) = to_tensors(Xa, ya), to_tensors(Xv, yv), to_tensors(Xte, yte)
    out_dir = os.path.join(WEIGHT_DIR, keyword)
    os.makedirs(out_dir, exist_ok=True)

    rows, best = [], None
    for r in ranks:
        set_seed(SEED)
        model = make_lora(r)
        trainable = [p for p in model.parameters() if p.requires_grad]
        model = fit(model, Xa, ya, Xv, yv, trainable, lr=lr)
        row = {"dataset": dataset, "features": cols, "rank": r, "params": sum(p.numel() for p in trainable),
               "train": accuracy(model, Xa, ya), "val": accuracy(model, Xv, yv)}
        rows.append(row)
        print(f"rank {r:>3} | params {row['params']:>9,} | train {row['train']:.4f} | val {row['val']:.4f}")
        if best is None or row["val"] > best["val"]:
            best, best_model = row, copy.deepcopy(model).merge_and_unload()

    best["test"] = accuracy(best_model, Xte, yte)
    best["state"] = best_model.state_dict()
    path = os.path.join(out_dir, "best.pt")
    torch.save(best["state"], path)
    np.save(os.path.join(out_dir, "features.npy"), np.array(cols))
    print(f"Best for {dataset}: rank {best['rank']} (val {best['val']:.4f}) -> test {best['test']:.4f}, saved {path}")
    return rows

adapt_results = rank_sweep_lite("kryptonite-40-adapt", "adapt", ranks=RANKS)

=== kryptonite-40-adapt: 2400 train / 800 val / 800 test ===
  + [24, 32] -> [24, 32]  val acc 0.5587
  + [3, 25] -> [3, 24, 25, 32]  val acc 0.5687
  + [10, 19] -> [3, 10, 19, 24, 25, 32]  val acc 0.5850
  + [8, 37] -> [3, 8, 10, 19, 24, 25, 32, 37]  val acc 0.6150
  + [17, 26] -> [3, 8, 10, 17, 19, 24, 25, 26, 32, 37]  val acc 0.6450
selected 10 features: [3, 8, 10, 17, 19, 24, 25, 26, 32, 37]
rank   1 | params     4,137 | train 0.7163 | val 0.6800
rank   2 | params     8,274 | train 0.7237 | val 0.6875
rank   4 | params    16,548 | train 0.7071 | val 0.6888
rank   8 | params    33,096 | train 0.7292 | val 0.6800
rank  16 | params    66,192 | train 0.7300 | val 0.6762
rank  32 | params   132,384 | train 0.7396 | val 0.6787
rank  64 | params   264,768 | train 0.7375 | val 0.6712
rank 128 | params   529,536 | train 0.7183 | val 0.6975
rank 256 | params 1,059,072 | train 0.7342 | val 0.6900
Best for kryptonite-40-adapt: rank 128 (val 0.6975) -> test 0.6625, saved ./weights/lora_fs_bin\a